## 1. Validate Taxi Fact Zone Keys

Load the Gold Taxi Daily Fact table and validate pickup location IDs before building the Zone Dimension.

Check the number of distinct pickup zones and ensure there are no NULL location IDs.

In [ ]:
df_fact = spark.table("dbo.gold_fact_taxi_daily_ready")

df_fact.select("PULocationID").distinct().count()

In [ ]:
from pyspark.sql import functions as F
df_fact.filter(
    F.col("PULocationID").isNull()
).count()

## 2. Validate Silver Taxi Zones

Load the Silver Taxi Zones reference table and validate its schema, row count, unique LocationIDs, and NULL values.

In [ ]:
df_zones = spark.table("dbo.silver_taxi_zones")

df_zones.printSchema()
df_zones.show(5, truncate=False)

In [ ]:
print("Total rows:", df_zones.count())

print(
    "Unique LocationIDs:",
    df_zones.select("LocationID").distinct().count()
)    

print(
    "NULL LocationIDs:",
    df_zones.filter(F.col("LocationID").isNull()).count()
)

## 3. Check Missing Zones

Use a LEFT ANTI JOIN to identify pickup location IDs in the Gold Taxi Fact that do not exist in the Silver Taxi Zones reference table.

In [ ]:
df_missing_zones = df_fact.select("PULocationID").distinct().join(
    df_zones,
    F.col("PULocationID") == F.col("LocationID"),
    "left_anti"
)

df_missing_zones.show()
print("Missing zones:", df_missing_zones.count())

## 4. Build Gold Zone Dimension

Create the Gold Zone Dimension from the validated Silver Taxi Zones reference table.

Retain the location ID, borough, zone name, and service zone for geographical analysis.

In [ ]:
df_dim_zone = df_zones.select(
    "LocationID",
    "Borough",
    "Zone",
    "service_zone"
)

df_dim_zone.show(5, truncate=False)

In [ ]:
print("Total rows:", df_dim_zone.count())

## 5. Save and Verify Gold Zone Dimension

Save the validated Zone Dimension as a Delta table in the Gold layer and verify the persisted data.

In [ ]:
df_dim_zone.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbo.gold_dim_zone")

In [ ]:
df_saved = spark.table("dbo.gold_dim_zone")

df_saved.show(5, truncate=False)
print("Saved rows:", df_saved.count())

## 6. Validate Fact-to-Zone Join

Perform a LEFT JOIN between the Gold Taxi Fact and Zone Dimension using location IDs.

Verify that geographical attributes are correctly mapped and that the join preserves the original fact row count.

In [ ]:
df_joined = df_fact.join(
    df_dim_zone,
    F.col("PULocationID") == F.col("LocationID"),
    "left"
)

df_joined.select(
    "trip_date",
    "PULocationID",
    "Borough",
    "Zone",
    "daily_trips"
).show(5, truncate=False)

In [ ]:
print("Fact rows:", df_fact.count())
print("Joined rows:", df_joined.count())